# ☕ Qehwa AI — پشتو چیٹ

**طریقہ:**
1. اوپر مینو میں **Runtime** → **Run all** دبائیں
2. تھوڑا انتظار کریں (ماڈل ڈاؤن لوڈ ہوگا، پہلی بار 5-10 منٹ)
3. آخر میں ایک چیٹ لنک ملے گا — اسے کھول کر پشتو/اردو/انگریزی میں سوال پوچھیں

نوٹ: یہ گوگل کا مفت کمپیوٹر استعمال کرتا ہے، اس لیے کبھی کبھار سست ہو سکتا ہے۔

In [ ]:
# ضروری سافٹ ویئر انسٹال کریں
!pip install -q transformers accelerate bitsandbytes gradio

In [ ]:
# Qehwa ماڈل لوڈ کریں (4-bit — مفت GPU پر چلتا ہے)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "junaid008/qehwa-pashto-llm"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("ٹوکنائزر لوڈ ہو رہا ہے...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("ماڈل لوڈ ہو رہا ہے... (5-10 منٹ لگ سکتے ہیں)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

ALPACA_TEMPLATE = """Below is an instruction in Pashto or English. Write a detailed response in Pashto.

### Instruction:
{}

### Response:
{}"""

def generate(prompt):
    inputs = tokenizer(ALPACA_TEMPLATE.format(prompt, ""), return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

print("✅ ماڈل تیار ہے!")
print(generate("سلام"))

In [ ]:
# چیٹ انٹرفیس شروع کریں
import gradio as gr

def chat(message, history):
    return generate(message)

demo = gr.ChatInterface(
    fn=chat,
    title="☕ Qehwa AI — پشتو چیٹ",
    description="پشتو، اردو یا انگریزی میں سوال لکھیں — جواب پشتو میں ملے گا",
)
demo.launch(share=True)